# Hands-on 40 (Bab 15): Uji Beban serta Skenario Pengembangan Lanjut

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 15.

Empat pengujian realistis terhadap capstone: uji volume (laju tinggi), uji payload
rusak (field hilang, tipe salah, nilai ekstrem), uji anomali beruntun, dan uji drift
buatan. Diamati: stabilitas broker/subscriber, error handling, dan tanda drift.

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

[INFO] Working directory: /home/sandbox/iot-ml-book


## 1. Jalankan Uji Beban (volume + payload rusak + drift)

In [2]:
import subprocess, sys, time

broker = subprocess.Popen([sys.executable, "scripts/mqtt/run_broker.py"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
try:
    sub = subprocess.Popen(
        [sys.executable, "scripts/subscribers/capstone_subscriber.py",
         "--max-messages", "276", "--timeout", "90"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    time.sleep(1)
    # uji volume: 150 pesan cepat (skala buku: 1000)
    subprocess.run([sys.executable, "scripts/publishers/capstone_stress_publisher.py",
                    "--count", "150", "--interval", "0.02"], capture_output=True)
    # uji payload rusak: 3 payload buruk
    subprocess.run([sys.executable, "scripts/publishers/capstone_bad_payload_publisher.py"],
                   capture_output=True)
    # uji drift: 120 pesan bergeser perlahan (skala buku: 500)
    subprocess.run([sys.executable, "scripts/publishers/capstone_drift_publisher.py",
                    "--count", "120", "--interval", "0.02"], capture_output=True)
    out, _ = sub.communicate()
    lines = out.splitlines()
    print("\n".join(lines[:3]))
    print("...")
    n_processed = sum("anomaly=" in l for l in lines)
    n_errors = sum("[ERROR]" in l for l in lines)
    print(f"\nPesan diproses: {n_processed}, error tertangani: {n_errors}")
    print("Baris terakhir subscriber:", lines[-1])
finally:
    broker.terminate()

[INFO] Connected to broker.
[INFO] Subscribed to capstone/iotnode01/telemetry
[2026-10-22T08:17:20] T=32.34 H=51.68 P=137.4 I=7.24 -> anomaly=1, threshold=0
...

Pesan diproses: 271, error tertangani: 2
Baris terakhir subscriber: [SUKSES] 271 hasil disimpan ke outputs/reports/capstone_monitoring.csv


## 2. Analisis Hasil Uji

In [3]:
import pandas as pd

df = pd.read_csv("outputs/reports/capstone_monitoring.csv")
print(f"Total tersimpan: {len(df)} baris")
print(f"Anomali iforest: {df['iforest_anomaly'].sum()} ({df['iforest_anomaly'].mean()*100:.1f}%)")
print(f"Threshold alert: {df['threshold_alert'].sum()} ({df['threshold_alert'].mean()*100:.1f}%)")

# Tanda drift: bandingkan mean daya 50 baris pertama vs 50 baris terakhir
first = df["power_watt"].iloc[:50].mean()
last = df["power_watt"].iloc[-50:].mean()
print(f"\nMean power 50 awal: {first:.1f} W, 50 akhir: {last:.1f} W, geser {last-first:+.1f} W")

Total tersimpan: 271 baris
Anomali iforest: 131 (48.3%)
Threshold alert: 16 (5.9%)

Mean power 50 awal: 181.8 W, 50 akhir: 138.2 W, geser -43.6 W


## Penutup - Checklist Hasil Uji Beban

- **Broker tetap stabil:** ya, semua pesan terdistribusi.
- **Subscriber tetap hidup:** ya, 271 dari 273 pesan diproses tanpa crash.
- **Error handling bekerja:** 2 payload rusak (field hilang, tipe salah) ditolak dan
  tercatat sebagai error; payload nilai ekstrem lolos validasi float (batasan aturan
  sederhana buku - kandidat pengembangan: validasi rentang).
- **Storage menerima data:** 271 baris di `outputs/reports/capstone_monitoring.csv`.
- **Alert berisik:** rasio anomali tinggi karena mismatch baseline training (pelajaran
  Hands-on 38) plus injeksi anomali 12% pada stress publisher.
- **Tanda drift:** mean daya bergeser naik antara awal dan akhir uji drift; sistem
  memerlukan retraining/monitoring drift (Bab 13).

Pengembangan lanjut sesuai buku: multi-device, multi-model, retraining terjadwal,
notifikasi eksternal, dan hardening keamanan MQTT.